# 데이터 불러오기

In [ ]:
!pip install --upgrade konlpy
!pip install soynlp
!pip install git+https://github.com/haven-jeon/PyKoSpacing.git

In [ ]:
!pip install --upgrade --force-reinstall --no-cache-dir gensim pyLDAvis

In [ ]:
# Colab에서 Mecab 실행 위해 필요
!git clone https://github.com/SOMJANG/Mecab-ko-for-Google-Colab.git
%cd Mecab-ko-for-Google-Colab
!bash install_mecab-ko_on_colab_light_220429.sh

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

In [ ]:
negative_df = pd.read_csv('/content/drive/MyDrive/구해줘! 잡스/감정분석_결과.csv', encoding='utf-8')

In [ ]:
negative_df

In [ ]:
negative_df = negative_df[negative_df['predicted_emotion'] == 'negative']

In [ ]:
negative_df['predicted_emotion'].value_counts()

# 전처리

In [ ]:
# 데이터 처리
import numpy as np

# 시각화
import matplotlib.pyplot as plt
import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

# 텍스트 벡터화 & 차원 축소
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.decomposition import TruncatedSVD, LatentDirichletAllocation

# 진행 표시줄
from tqdm import tqdm

# Gensim 기반 토픽 모델링
from gensim.corpora import Dictionary
from gensim.models import LdaModel, CoherenceModel
from gensim.models.phrases import Phrases, Phraser

# 병렬 처리
from joblib import Parallel, delayed

# Standard library imports
import warnings
import re
import ast
from datetime import datetime, timedelta

# Data manipulation
import numpy as np
import pandas as pd
from google.colab import drive

# Visualization
import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
from konlpy.tag import Okt, Mecab
from pykospacing import Spacing
from soynlp.normalizer import repeat_normalize, emoticon_normalize

# Suppress warnings
warnings.filterwarnings('ignore')

In [ ]:
# 형태소 분석 및 어간 추출 -> Mecab
mecab = Mecab()
negative_df['morphs'] = negative_df['emotion_text'].apply(lambda x: mecab.morphs(x))

In [ ]:
# 불용어 제거
stopwords = stopwords = {
    # 기본 기능어
    '것', '수', '등', '더', '의',
    # 조사·어미
    '은', '는', '이', '가', '을', '를',
    '에', '에서', '부터', '까지', '으로', '로', '에게',
    # 접속사·전후관계
    '그리고', '하지만', '그러나', '그런데', '그래서',
    # 대명사·지시어
    '나', '우리', '저희', '너', '너희', '이곳', '여기', '저기',
    # 기타 자주 쓰이는 문법어
    '때문', '그러므로', '및',
    # 간단한 감탄사/의성어
    '아', '휴', '아이쿠', '헐', '헉',
    # 추가 사용자 정의 불용어
    '아이구', '아이고', '어', '따라', '의해', '뿐이다', '의거하여', '근거하여', '입각하여', '기준으로',
    '예하면', '예를 들면', '예를 들자면', '저', '소인', '소생', '지말고', '하지마', '하지마라', '다른', '물론',
    '또한', '비길수 없다', '해서는 안된다', '뿐만 아니라', '만이 아니다', '만은 아니다', '막론하고', '관계없이',
    '그치지 않는다', '든간에', '논하지 않는다', '따지지 않는다', '설사', '비록', '더라도', '아니면', '만 못하다',
    '하는 편이 낫다', '불문하고', '향하여', '향해서', '향하다', '쪽으로', '틈타', '이용하여', '타다', '오르다',
    '제외하고', '이 외에', '이 밖에', '하여야', '비로소', '한다면 몰라도', '외에도', '기점으로', '할 생각이다',
    '하려고하다', '이리하여', '그리하여', '그렇게 함으로써', '일때', '할때', '앞에서', '중에서', '보는데서',
    '으로써', '로써', '해야한다', '일것이다', '반드시', '할줄알다', '할수있다', '할수있어', '임에 틀림없다',
    '한다면', '등등', '제', '겨우', '단지', '다만', '할뿐', '딩동', '댕그', '대해서', '대하여', '대하면', '훨씬',
    '얼마나', '얼마만큼', '얼마큼', '남짓', '여', '얼마간', '약간', '다소', '좀', '조금', '다수', '몇', '얼마',
    '지만', '하물며', '이외에도', '대해 말하자면', '다음에', '반대로', '반대로 말하자면', '이와 반대로',
    '바꾸어서 말하면', '바꾸어서 한다면', '만약', '그렇지않으면', '까악', '툭', '딱', '삐걱거리다', '보드득',
    '비걱거리다', '꽈당', '응당', '에 가서', '각', '각각', '여러분', '각종', '각자', '제각기', '하도록하다',
    '와', '과', '고로', '한 까닭에', '하기 때문에', '거니와', '이지만', '관하여', '관한', '과연', '실로',
    '아니나다를가', '생각한대로', '진짜로', '한적이있다', '하곤하였다', '하', '하하', '허허', '아하', '거바',
    '오', '왜', '어째서', '무엇때문에', '어찌', '하겠는가', '무슨', '어디', '어느곳', '더군다나', '더욱이는',
    '어느때', '언제', '야', '이봐', '어이', '여보시오', '흐흐', '흥', '헉헉', '헐떡헐떡', '영차', '여차',
    '어기여차', '끙끙', '아야', '앗', '콸콸', '졸졸', '좍좍', '뚝뚝', '주룩주룩', '솨', '우르르', '그래도',
    '바꾸어말하면', '바꾸어말하자면', '혹은', '혹시', '답다', '때가 되어', '즉', '지든지', '설령', '가령',
    '하더라도', '할지라도', '일지라도', '하마터면', '인젠', '이젠', '된바에야', '된이상', '어찌됏든',
    '그위에', '게다가', '점에서 보아', '비추어 보아', '고려하면', '하게될것이다', '비교적', '보다더', '비하면',
    '시키다', '하게하다', '할만하다', '연이서', '이어서', '잇따라', '뒤따라', '뒤이어', '결국', '의지하여',
    '기대여', '통하여', '자마자', '더욱더', '불구하고', '얼마든지', '마음대로', '주저하지 않고', '곧', '즉시',
    '바로', '당장', '하자마자', '밖에 안된다', '하면된다', '그래', '그렇지', '요컨대', '다시 말하자면',
    '구체적으로', '말하자면', '시작하여', '시초에', '이상', '허', '허걱', '바와같이', '해도좋다', '해도된다',
    '더구나', '와르르', '팍', '퍽', '펄렁', '동안', '이래', '하고있었다', '이었다', '로부터', '같이', '더불어',
    '마저', '양자', '모두', '가까스로', '즈음하여', '해봐요', '습니까', '말할것도 없고', '개의치않고',
    '하는것만 못하다', '하는것이 낫다', '매', '매번', '들', '모', '어느것', '어느', '갖고말하자면', '어느쪽',
    '어느해', '어느 년도', '라 해도', '언젠가', '어떤것', '저쪽', '저것', '그때', '그럼', '그러면', '요만한걸',
    '저것만큼', '그저', '할 줄 안다', '할 힘이 있다', '당신', '설마', '차라리', '놀라다', '상대적으로 말하자면',
    '마치', '아니라면', '쉿', '그렇지 않으면', '안 그러면', '아니었다면', '하든지', '이라면', '좋아', '알았어',
    '그만이다', '어쩔수 없다', '하나', '일', '일반적으로', '일단', '오자마자', '이렇게되면', '이와같다면', '전부',
    '한마디', '한항목', '근거로', '하기에', '아울러', '않기 위해서', '이르기까지', '로 인하여', '까닭으로',
    '이유만으로', '이로 인하여', '이 때문에', '결론을 낼 수 있다', '으로 인하여', '알 수 있다',
    '합니다', '습니다', '된다', '된다면', '되어', '했다', '합니다', '된다면',
    '그냥', '거기', '거의', '같다', '너무', '매우', '많이', '진짜', '완전',
    '인데', '는데', '네요', '어요', '아요', '입니다', '있습니다', '없습니다',
    '나요', '가요', '어요', '아요', '죠', '잖아', '거든요', '네', '군요', '지요',
    '했나요', '가나요', '사나요', '먹나요', '하나요', '되나요', '받나요', '알나요', '있나요', '없나요',
    '되었나요', '오나요', '주나요', '나오나요',
    '말하나요', '알겠나요', '끝났나요', '시작하나요', '가나요', '보나요',
    '하려나요', '괜찮나요', '필요하나요', '필요없나요'
}

In [ ]:
negative_df['tokens'] = negative_df['morphs'].apply(lambda tokens: [t for t in tokens if t not in stopwords and len(t) > 1])

In [ ]:
negative_df['tokens'].head()

In [ ]:
# 토큰을 문자열로 결합
negative_df['processed'] = negative_df['tokens'].apply(lambda tokens: ' '.join(tokens))

In [ ]:
negative_df['processed'].head()

In [ ]:
# 사용자 정의 복합어 쌍 리스트
COMPOUND_PAIRS = [
    ('자',   '소서'),
    ('자소', '서'),
    ('인적', '성'),
    ('인', '적성')
]

# 명사 태그
NOUN_TAGS = {"NNG", "NNP", "NNB", "NNBC", "NP"}

def merge_compound_morphs(morphs, pairs=COMPOUND_PAIRS, pos="NNG"):
    """
    morphs: [(word, tag), …]
    pairs: [('자','소서'), ('자소','서'), ('인적','성'), …]
    pos: 합쳐진 토큰에 부여할 품사 태그
    """
    merged = []
    i = 0
    while i < len(morphs):
        if i + 1 < len(morphs):
            matched = False
            for a, b in pairs:
                if morphs[i][0] == a and morphs[i+1][0] == b:
                    merged.append((a + b, pos))
                    i += 2
                    matched = True
                    break
            if matched:
                continue
        merged.append(morphs[i])
        i += 1
    return merged

def clean_for_topic(text):
    # HTML·URL 제거
    text = re.sub(r'<.*?>|http\S+', ' ', text)
    # 한글·공백 외 제거
    text = re.sub(r'[^가-힣\s]', ' ', text)
    # 공백 단일화
    text = re.sub(r'\s+', ' ', text).strip()
    # 형태소 분석
    morphs = mecab.pos(text)
    # 복합어 합치기
    morphs = merge_compound_morphs(morphs)
    # 명사만 추출
    tokens = [w for w, t in morphs if t in NOUN_TAGS]
    # 불용어·길이 필터
    tokens = [t for t in tokens if t not in stopwords and len(t) > 1]
    # 문자열로 합쳐서 반환
    return ' '.join(tokens)

In [ ]:
negative_df['emotion_text'] = negative_df['emotion_text'].apply(clean_for_topic)

In [ ]:
negative_df = negative_df.rename(columns={'emotion_text': 'negative_text'})

In [ ]:
negative_df

# 토픽 모델링

In [ ]:
negative_df = negative_df.dropna(subset=['negative_text'])
negative_df = negative_df[negative_df['negative_text'].str.strip().astype(bool)]

# 텍스트를 띄어쓰기 기준으로 토큰화
texts = negative_df['negative_text'].dropna().str.split().tolist()

# Dictionary 및 Corpus 생성 (Gensim용)
dictionary = Dictionary(texts)
corpus     = [dictionary.doc2bow(text) for text in texts]

In [ ]:
topic_range = list(range(3, 11))
perplexity_scores = []
coherence_scores  = []

for nt in tqdm(topic_range, desc="Evaluating topics"):
    model = LdaModel(
        corpus=corpus,
        id2word=dictionary,
        num_topics=nt,
        random_state=42,
        passes=10,
        iterations=100
    )
    coh  = CoherenceModel(
        model=model,
        texts=texts,
        dictionary=dictionary,
        coherence='c_v'
    ).get_coherence()
    logp = model.log_perplexity(corpus)
    perp = np.exp2(-logp)

    coherence_scores.append(coh)
    perplexity_scores.append(perp)

# 시각화
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(topic_range, perplexity_scores, 'o-', label='Perplexity')
ax1.set_title('Perplexity by Number of Topics')
ax1.set_xlabel('Number of Topics')
ax1.set_ylabel('Perplexity')
ax1.grid(True)

ax2.plot(topic_range, coherence_scores, 'o-', color='tab:green', label='Coherence (c_v)')
ax2.set_title('Coherence by Number of Topics')
ax2.set_xlabel('Number of Topics')
ax2.set_ylabel('Coherence Score')
ax2.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation

vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words=list(stopwords))
X = vectorizer.fit_transform(negative_df['negative_text'])

lda = LatentDirichletAllocation(n_components=4, random_state=42)
lda.fit(X)

# 토픽별 주요 키워드 출력
words = vectorizer.get_feature_names_out()
for i, topic in enumerate(lda.components_):
    top_words = [words[j] for j in topic.argsort()[:-7:-1]]
    print(f"토픽 {i}: {' '.join(top_words)}")

# 문서별 dominant topic 저장
negative_df['dominant_topic'] = lda.transform(X).argmax(axis=1)

#. pyLDAvis 시각화 (Gensim 모델 기반)
pyLDAvis.enable_notebook()

# Gensim용 LDA 재학습 (선택된 토픽 수 기준, 예: 3개)
model = LdaModel(
    corpus=corpus,
    id2word=dictionary,
    num_topics=3,
    random_state=42,
    passes=10,
    iterations=100
)

lda_vis = gensimvis.prepare(
    topic_model=model,
    corpus=corpus,
    dictionary=dictionary,
    sort_topics=False
)
lda_vis

# 저장
pyLDAvis.save_html(lda_vis, '/content/negative_lda_visualization.html')

| 토픽 번호    | 주요 키워드                  | 해석                                               | 부정 감정의 원인                  |
| -------- | ----------------------- | ------------------------------------------------ | -------------------------- |
| **토픽 0** | 면접, 준비, 서류, 생각, 회사, 인적성 | **채용 과정 초반 단계**에서의 막막함<br>(서류 탈락, 인적성 부담, 면접 부담) | 불확실한 기준, 반복 탈락, 준비 피로감     |
| **토픽 1** | 취업, 기업, 사람, 신입, 직무, 서류  | **지원 대상 기업 선정과 직무 고민**<br>→ 사람들과의 비교, 경쟁 심리      | 비교/열등감, '나만 뒤처지는 것 같은' 느낌  |
| **토픽 2** | 기업, 생각, 취업, 연봉, 공기업, 서울 | **취업 후 삶에 대한 현실적 불안**<br>연봉, 지역, 공기업 경쟁 등        | 낮은 연봉, 지역 차별, ‘이래도 되는가’ 불안 |
| **토픽 3** | 인턴, 준비, 직무, 경험, 스펙, 감사  | **스펙 부족에 대한 스트레스**<br>→ 인턴 경험, 감사 활동, 직무 경험 부족   | 경험 부족, 준비 부족, 자기비하         |


# bertopic

In [ ]:
!pip install bertopic

In [ ]:
from bertopic import BERTopic

# 텍스트 리스트 준비
docs = negative_df['negative_text'].tolist()

# 모델 학습
topic_model = BERTopic(min_topic_size=5, language='multilingual', nr_topics=4)  # 한국어 지원
topics, probs = topic_model.fit_transform(docs)

In [ ]:
# 주제 확인
topic_model.get_topic_info()
topic_model.get_topic(0)

topic_model.visualize_barchart(top_n_topics=4)  # 이미 확인함
topic_model.visualize_documents(docs)  # 문서 기반 2D 시각화 (잘 작동함)

In [ ]:
topic_model.visualize_barchart(top_n_topics=None)

In [ ]:
topic_model.get_topic_info()